# Model Tester

In [24]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import os
import sys

os.environ["KERAS_BACKEND"] = "tensorflow"
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
# os.environ["XLA_FLAGS"] = (
#     "--xla_gpu_cuda_data_dir=/hpc/mp/apps/nvidia/hpc_sdk/23.7/Linux_x86_64/23.7/cuda"
# )

In [26]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)

Conda environment: deepsphere
Python executable: /work/users/stevensonb/.conda/envs/deepsphere/bin/python


## Setup

In [ ]:
import sys
import time
import logging

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras

from keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from keras.optimizers import Adam
from keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule
from keras.metrics import RootMeanSquaredError

# sys.path.append(os.path.join(os.getcwd(), ".."))
from mlpng.models import AutoModel
from mlpng.models.modelcore import get_model_class
from mlpng.utils import (
    setup_logging,
    load_data,
    get_fisher,
    plot_predictions,
    plot_histogram,
    print_errors,
    plot_metrics,
    WarmupLearningRate,
    AttentionSchedule,
)

import deepsphere
from deepsphere import HealpyGCNN
from deepsphere import healpy_layers as hp_layer
import healpy as hp

# from deepsphere import HealpyGCNN

# tf.debugging.set_log_device_placement(true)

In [ ]:
# print(f"TensorFlow version: {tf.__version__}")
# print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
# print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")
print(f"keras version: {keras.__version__}")

keras version: 3.6.0


In [29]:
logger = setup_logging(__name__, level=logging.DEBUG)

In [ ]:
# Function to reset the TensorFlow session and strategy
# doesnt fully work for reasons i dont understand
def reset_session_and_strategy():
    keras.backend.clear_session()  # Clear the current TensorFlow session
    try:
        del strategy  # Delete the existing strategy
        del model
    except:
        pass

    return keras.distribute.MirroredStrategy()


def plot_lr(lr, steps):
    # Compute learning rates for each step
    learning_rates = lr(steps)

    # Plot the learning rate as a function of steps
    plt.figure(figsize=(10, 6))
    plt.loglog(steps, learning_rates)
    plt.xlabel("Steps")
    plt.ylabel("Learning Rate")
    plt.title("Learning Rate Schedule")
    plt.grid(True)
    plt.show()

## Configure

In [31]:
args = [
    "settings/n128.json",
    "--nsims",
    "100",
    "--narray",
    "1000",
    "--pols",
    "T",
    "--fnl_range",
    "-100",
    "100",
]

MAX_EPOCHS = 100
BATCH_SIZE = 128

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    # TensorBoard(log_dir=f"data/tensorboard/notebooks/{time.strftime('%Y%m%d-%H%M%S')}"),
    TerminateOnNaN(),
]

In [32]:
# from mlpng.utils.tf import try_init_wandb

# try_init_wandb(notes="model testing", tags=["notebook"], append_to=callbacks)

Right now only one model can be run at a time if you are using the superpod and multi-GPU. This seems to be an issue with CUDA not respecting clear_session() and the GPU memory is not being released. I have tried to fix this a few times but nothing has worked, run the above cells and then pick the model you want to run below.

In [33]:
# check minst handwriting with spherical projection

## Deepsphere

In [ ]:
layers = [
    hp_layer.HealpyChebyshev(
        K=10, Fout=5, use_bias=True, use_bn=True, activation="relu"
    ),
    hp_layer.HealpyPool(p=1),
    hp_layer.HealpyChebyshev(
        K=10, Fout=5, use_bias=True, use_bn=True, activation="relu"
    ),
    hp_layer.HealpyPool(p=1),
    hp_layer.HealpyChebyshev(
        K=10, Fout=5, use_bias=True, use_bn=True, activation="relu"
    ),
    hp_layer.HealpyPool(p=1),
    hp_layer.HealpyChebyshev(K=10, Fout=2),
    keras.layers.Lambda(lambda x: tf.nn.softmax(tf.reduce_mean(x, axis=1), axis=-1)),
]

nside = 64
indices = np.arange(hp.nside2npix(nside))

keras.backend.clear_session()
model = HealpyGCNN(nside=nside, indices=indices, layers=layers, n_neighbors=20)
batch_size = 16
model.build(input_shape=(None, len(indices), 1))
model.summary(110)

Detected a reduction factor of 8.0, the input with nside 64 will be transformed to 8 during a forward pass. Checking for consistency with indices...
indices seem consistent...
kwargs: {}
kwargs: {}
kwargs: {}
kwargs: {}


Model: "healpy_gcnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                   ┃ Output Shape                        ┃             Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ chebyshev (Chebyshev)                          │ (None, 49152, 5)                    │                  65 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ healpy_pool (HealpyPool)                       │ (None, 12288, 5)                    │                   0 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ chebyshev_1 (Chebyshev)                        │ (None, 12288, 5)                    │                 265 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ healpy_pool_1 (HealpyPool)                     │ (None, 3072, 5)                     │                   0 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ chebyshev_2 (Chebyshev)                        │ (None, 3072, 5)                     │                 265 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ healpy_pool_2 (HealpyPool)                     │ (None, 768, 5)                      │                   0 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ chebyshev_3 (Chebyshev)                        │ (None, 768, 2)                      │                 100 │
├────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┤
│ lambda (Lambda)                                │ (None, 2)                           │                   0 │
└────────────────────────────────────────────────┴─────────────────────────────────────┴─────────────────────┘

 Total params: 695 (2.71 KB)

 Trainable params: 665 (2.60 KB)

 Non-trainable params: 30 (120.00 B)

NameError: name 'data' is not defined

In [ ]:
alm = get_model_class("ALM")(args)
fisher = get_fisher(alm.file)
fnl_scale = (alm.fnl_max - alm.fnl_min) / 2.0
scaled_std = 1 / np.sqrt(fisher) / fnl_scale
# print("Fisher:", fisher, "scaled std:", scaled_std)

ds = alm.init_dataset(
    shuffle=False,
    shuffle_buffer=None,
    seed=0,
    batch_size=BATCH_SIZE,
    cache=True,
    normalize=True,
    channels_last=True,
    fnl_scale=fnl_scale,
)
train_ds, test_ds, val_ds = ds.get_split(0.5, 0.4, 0.1)

# x_raw = np.concatenate([data["class1"], data["class2"]]).astype(np.float32)[..., None]
# y_raw = np.concatenate([np.zeros(nclass), np.ones(nclass)]).astype(np.float32)

# np.random.RandomState(11).shuffle(x_raw)
# np.random.RandomState(11).shuffle(y_raw)

# x_train, x_test = np.split(x_raw, indices_or_sections=[150])
# y_train, y_test = np.split(y_raw, indices_or_sections=[150])


model.compile(
    optimizer=keras.optimizers.Adam(0.1),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

history = model.fit(
    train_ds,
    batch_size=batch_size,
    epochs=20,
    validation_data=val_ds,
)

14-Nov-24 15:32:30 - mlpng.core - INFO - Parsing CLI args: ['settings/n128.json', '--nsims', '100', '--narray', '1000', '--pols', 'T', '--fnl_range', '-100', '100']
14-Nov-24 15:32:30 - mlpng.core - INFO - Loading settings from file 'settings/n128.json'
14-Nov-24 15:32:30 - mlpng.core - DEBUG - Forcing setting 'nsims' to 100 due to CLI
14-Nov-24 15:32:30 - mlpng.core - DEBUG - Forcing setting 'narray' to 1000 due to CLI
14-Nov-24 15:32:30 - mlpng.core - DEBUG - Forcing setting 'fnl_range' to [-100.0, 100.0] due to CLI
14-Nov-24 15:32:30 - mlpng.core - DEBUG - Forcing setting 'pols' to ['T'] due to CLI
14-Nov-24 15:32:30 - mlpng.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 67.66, 'As': 2.1056e-09, 'ns': 0.9665, 'ombh2': 0.02242, 'omch2': 0.11933, 'tau': 0.0561, 'pivot_scalar': 0.05} (default: {'As': 2.13e-09, 'ns': 0.9624, 'pivot_scalar': 0.05})
14-Nov-24 15:32:30 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.1056e-09
14-Nov-24 15:32:30 - mlp

TypeError: `generator` must be a Python callable.

## ALM Model

In [ ]:
model = get_model_class("ALM")(args)
fisher = get_fisher(model.file)
fnl_scale = (model.fnl_max - model.fnl_min) / 2.0
scaled_std = 1 / np.sqrt(fisher) / fnl_scale
# print("Fisher:", fisher, "scaled std:", scaled_std)

ds = model.init_dataset(
    shuffle=False,
    shuffle_buffer=None,
    seed=0,
    batch_size=BATCH_SIZE,
    cache=True,
    normalize=True,
    channels_last=True,
    fnl_scale=fnl_scale,
)
train_ds, test_ds, val_ds = ds.get_split(0.5, 0.4, 0.1)

learning_rate = ExponentialDecay(1e-3, 100, 0.95, staircase=True)


# learning_rate = WarmupLearningRate(
#     warmup_learning_rate=1e-3,
#     warmup_steps=8000,
#     warmup_scale=1.5,
#     warmup_scale_steps=10,
#     warmed_learning_rate="auto",
#     decay_steps=10000,
#     decay_rate=0.95,
#     staircase=True,
# )
# learning_rate = AttentionSchedule(d_model=383, warmup_steps=8000)


def cLoss(lf, boundary=1e-2):
    """loss with a inner boundary to repulse 0 guesses, but an epsilon to allow for exact guesses to be counted as correct"""

    def loss(y_true, y_pred):
        # error = tf.abs(y_true - y_pred)
        loss = lf(y_true, y_pred)
        # add a bump on 0 guesses to repulse them
        # loss(b) = b^2 / 2, cos -> 1 as y_pred -> 0
        # bc: need loss' = bump' at x=b, loss'=b, bump'(x=b)= (b^2/2 (1+cos(pi/2*b/b))' = (b^2/2)'
        bump = loss * (1 + 1 * tf.cos(np.pi / 2 * y_pred / boundary))
        return tf.where(tf.greater_equal(tf.abs(y_pred), boundary), loss, bump)

    return loss


strategy = reset_session_and_strategy()
with strategy.scope():
    metrics = [RootMeanSquaredError(), "mse", "mae"]
    opt = Adam(learning_rate)
    model.make_model()
    model.compile(
        optimizer=opt,
        loss=cLoss(
            lf=keras.losses.Huber(scaled_std, reduction=keras.losses.Reduction.NONE)
        ),
        metrics=metrics,
    )

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
# 0.0242

In [ ]:
# model.evaluate(test_ds, verbose=1)
preds = model.predict(test_ds, verbose=1).flatten() * fnl_scale
truth = np.concatenate([y.numpy() for _, y in test_ds]).flatten() * fnl_scale

print_errors(truth, preds, fisher)
plot_metrics(history, metrics=["loss"])  # , "root_mean_squared_error", "mae"] )
plot_predictions(truth, preds, fisher=fisher, show=True)
plot_histogram(truth, preds, show=True)

## Isensee Model

In [ ]:
model = AutoModel(args + ["--model", "ISENSEE"])
fisher = get_fisher(model.file)
fnl_scale = (model.fnl_max - model.fnl_min) / 2.0
scaled_std = 1 / np.sqrt(fisher) / fnl_scale

ds = model.init_dataset(
    shuffle=False,
    shuffle_buffer=None,
    seed=0,
    batch_size=BATCH_SIZE,
    cache=True,
    normalize=True,
    channels_last=True,
)

train_ds, test_ds, val_ds = ds.get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 1000, 0.96, staircase=True)
# learning_rate = WarmupLearningRate(warmup_steps=warmup_steps)
# plot_lr(learning_rate, np.arange(warmup_steps * MAX_EPOCHS))


def cLoss(lf, boundary=1e-2):
    def loss(y_true, y_pred):
        # error = tf.abs(y_true - y_pred)
        loss = lf(y_true, y_pred)
        # add a bump on 0 guesses to repulse them
        # loss(b) = b^2 / 2, cos -> 1 as y_pred -> 0
        # bc: need loss' = bump' at x=b, loss'=b, bump'(x=b)= (b^2/2 (1+cos(pi/2*b/b))' = (b^2/2)'
        bump = loss * (1 + 1 * tf.cos(np.pi / 2 * y_pred / boundary))
        return tf.where(tf.greater_equal(tf.abs(y_pred), boundary), loss, bump)

    return loss


strategy = reset_session_and_strategy()
with strategy.scope():
    metrics = [RootMeanSquaredError()]
    opt = Adam(learning_rate)
    model.make_model()

    model.compile(
        optimizer=opt,
        loss=cLoss(
            keras.losses.Huber(scaled_std, reduction=keras.losses.Reduction.NONE)
        ),
        metrics=metrics,
    )
    # model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

fisher = get_fisher(model.file)

model.evaluate(test_ds, verbose=1)
preds = model.predict(test_ds, verbose=1).flatten()
truth = np.concatenate([y.numpy() for _, y in test_ds])

plot_metrics(history, metrics=["loss"])
plot_predictions(truth, preds, fisher=fisher, show=True)
plot_histogram(truth, preds, show=True)
print_errors(truth, preds, fisher)

## NAGARAJAPPA

In [ ]:
# model = get_model_class("NAGARAJAPPA")(args)
# ds = model.init_dataset(
#     shuffle=False,
#     shuffle_buffer=None,
#     seed=0,
#     batch_size=BATCH_SIZE,
#     cache=True,
#     normalize=True,
#     channels_last=True,
# )
# train_ds, test_ds, val_ds = ds.get_split(0.8, 0.1, 0.1)

# warmup_steps = len(train_ds) // BATCH_SIZE
# learning_rate = ExponentialDecay(1e-2, warmup_steps, 0.5, staircase=True)

# plot_lr(learning_rate, np.arange(warmup_steps * MAX_EPOCHS))

# strategy = reset_session_and_strategy()
# with strategy.scope():
#     metrics = [RootMeanSquaredError()]
#     opt = Adam(learning_rate)
#     model.make_model()
#     model.compile(optimizer=opt, loss="mse", metrics=metrics)
#     model.summary()

# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=MAX_EPOCHS,
#     callbacks=callbacks,
#     verbose=0,
# )

# fisher = get_fisher(model.file)

# model.evaluate(test_ds, verbose=1)
# preds = model.predict(test_ds, verbose=1).flatten()
# truth = np.concatenate([y.numpy() for _, y in test_ds])

# plot_metrics(history, metrics=["loss"])
# plot_predictions(truth, preds, fisher=fisher, show=True)
# plot_histogram(truth, preds, show=True)
# print_errors(truth, preds, fisher)